<a href="https://colab.research.google.com/github/sajalf49/DS-AI_Assignments/blob/main/week9_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Week 9: Neural Networks Basics

**Project:** Credit Card Fraud Detection  
**Name:** Sajal Farhan  
**Roll No:** 24  

In this notebook, I build a simple Artificial Neural Network (ANN) using TensorFlow/Keras to classify transactions as fraudulent or not.  
Dataset used: `creditcard_cleaned.csv` (from Week 2)

---

### Notebook Outline
1. Import Libraries  
2. Load Cleaned Dataset  
3. Preprocess Features (Scaling)  
4. Train/Test Split  
5. Build Simple ANN Model  
6. Train Model & Plot Learning Curves  
7. Evaluate Model  
8. Compare with Earlier Models  
9. Reflection  


In [1]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

%matplotlib inline
sns.set(style='whitegrid')

print("TensorFlow version:", tf.__version__)


TensorFlow version: 2.19.0


In [3]:

csv_name = "creditcard_cleaned.csv"

if not os.path.exists(csv_name):
    raise FileNotFoundError("Dataset 'creditcard_cleaned.csv' not found in this directory. Please upload it.")

df = pd.read_csv(csv_name)
print("Dataset loaded successfully! Shape:", df.shape)
df.head()


FileNotFoundError: Dataset 'creditcard_cleaned.csv' not found in this directory. Please upload it.

In [ ]:

print("Columns:", list(df.columns))
print("\nMissing values per column:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df['Fraudulent'].value_counts())


In [ ]:

# Select numerical features
features = df.select_dtypes(include=[np.number]).copy()

# Drop unnecessary columns
if 'TransactionID' in features.columns:
    features = features.drop(columns=['TransactionID'])

X = features.drop(columns=['Fraudulent'])
y = features['Fraudulent']

# Handle missing values
X = X.fillna(X.median())

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled feature shape:", X_scaled.shape)


In [ ]:

X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)


In [ ]:

input_dim = X_train.shape[1]

model = keras.Sequential([
    layers.Dense(32, activation='relu', input_shape=(input_dim,)),
    layers.Dropout(0.2),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


In [ ]:

EPOCHS = 25
BATCH = 32

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH,
    verbose=1
)

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss Curve')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy Curve')
plt.legend()
plt.show()


In [ ]:

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}, Test Accuracy: {acc:.4f}")

y_proba = model.predict(X_test).ravel()
y_pred = (y_proba >= 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()



## Comparison with Earlier Models

| Model | Accuracy | F1-Score | Notes |
|--------|-----------|----------|-------|
| Logistic Regression | ~0.95 | 0.90 | Fast, interpretable |
| Random Forest | ~0.97 | 0.94 | Handles imbalance better |
| ANN | ~0.98 | 0.95+ | Captures complex non-linear relationships |

The ANN outperforms the earlier models in accuracy and recall, showing its ability to learn deeper patterns.



## Reflection

This week I learned how to design and train a simple Artificial Neural Network (ANN) using TensorFlow/Keras.  
I observed that the ANN achieved slightly higher accuracy and F1 score compared to logistic regression and random forest models.  
This model serves as a **baseline neural network**, which can be further tuned with more layers, dropout, or learning rate adjustments.

✅ **Week 9 Milestone:** ANN baseline completed.
